# Practical 2: Regression Analysis - Uber Fare Prediction
**Problem Statement:** Predict the price of an Uber ride from a given pickup point to the drop-off location.

**Tasks:**
1. Pre-process the dataset
2. Identify outliers
3. Check the correlation
4. Implement Linear, Ridge and Lasso regression
5. Evaluate and compare the models (R2, RMSE, etc.)

**Dataset:** https://www.kaggle.com/datasets/yasserh/uber-fares-dataset (file: `uber.csv`)

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

import warnings
warnings.filterwarnings("ignore")
%matplotlib inline

## 2. Load the Dataset
Download `uber.csv` from Kaggle and keep it in the same folder as this notebook.

In [ ]:
df = pd.read_csv("uber.csv")
df.head()

In [ ]:
print("Shape:", df.shape)
df.info()

In [ ]:
df.describe()

# Task 1: Pre-process the Dataset

### 1.1 Drop unnecessary columns

In [ ]:
# 'Unnamed: 0' is just an index and 'key' duplicates pickup_datetime
df = df.drop(columns=[c for c in ["Unnamed: 0", "key"] if c in df.columns])
df.columns.tolist()

### 1.2 Missing values and duplicates

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())

In [ ]:
df = df.dropna()
df = df.drop_duplicates()
print("Shape after removing missing values and duplicates:", df.shape)

### 1.3 Convert datetime and extract features

In [ ]:
df["pickup_datetime"] = pd.to_datetime(df["pickup_datetime"], errors="coerce")
df = df.dropna(subset=["pickup_datetime"])

df["year"] = df["pickup_datetime"].dt.year
df["month"] = df["pickup_datetime"].dt.month
df["day"] = df["pickup_datetime"].dt.day
df["hour"] = df["pickup_datetime"].dt.hour
df["day_of_week"] = df["pickup_datetime"].dt.dayofweek

df[["pickup_datetime", "year", "month", "day", "hour", "day_of_week"]].head()

### 1.4 Compute trip distance from coordinates (Haversine formula)
The fare depends mostly on distance, but the dataset only has latitude and longitude, so distance in km is calculated.

In [ ]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

df["distance_km"] = haversine(df["pickup_latitude"], df["pickup_longitude"],
                              df["dropoff_latitude"], df["dropoff_longitude"])
df[["distance_km", "fare_amount"]].describe()

# Task 2: Identify Outliers

### 2.1 Visualize outliers with boxplots

In [ ]:
cols = ["fare_amount", "distance_km", "passenger_count"]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, c in zip(axes, cols):
    sns.boxplot(y=df[c], ax=ax, color="lightsteelblue")
    ax.set_title(f"Boxplot of {c}")
plt.tight_layout()
plt.show()

### 2.2 Count outliers with the IQR method

In [ ]:
def iqr_outliers(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).sum(), lower, upper

for c in ["fare_amount", "distance_km"]:
    n, lo, hi = iqr_outliers(df[c])
    print(f"{c:15s} -> outliers: {n:6d}  (valid range: {lo:.2f} to {hi:.2f})")

### 2.3 Find invalid / impossible values

In [ ]:
print("Fare <= 0                 :", (df["fare_amount"] <= 0).sum())
print("Passenger count == 0 or >6:", ((df["passenger_count"] <= 0) | (df["passenger_count"] > 6)).sum())
print("Invalid latitude          :", ((df["pickup_latitude"].abs() > 90) | (df["dropoff_latitude"].abs() > 90)).sum())
print("Invalid longitude         :", ((df["pickup_longitude"].abs() > 180) | (df["dropoff_longitude"].abs() > 180)).sum())
print("Distance == 0             :", (df["distance_km"] == 0).sum())
print("Distance > 100 km         :", (df["distance_km"] > 100).sum())

### 2.4 Remove outliers
Remove impossible values, then keep only realistic New York trips.

In [ ]:
before = len(df)

df = df[(df["fare_amount"] > 0) & (df["fare_amount"] <= 100)]
df = df[(df["passenger_count"] >= 1) & (df["passenger_count"] <= 6)]
df = df[(df["pickup_latitude"].between(40, 42)) & (df["dropoff_latitude"].between(40, 42))]
df = df[(df["pickup_longitude"].between(-75, -72)) & (df["dropoff_longitude"].between(-75, -72))]
df = df[(df["distance_km"] > 0) & (df["distance_km"] <= 60)]

print(f"Rows before: {before}")
print(f"Rows after : {len(df)}")
print(f"Removed    : {before - len(df)}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, c in zip(axes, cols):
    sns.boxplot(y=df[c], ax=ax, color="lightgreen")
    ax.set_title(f"{c} (after cleaning)")
plt.tight_layout()
plt.show()

# Task 3: Check the Correlation

In [ ]:
num_df = df.drop(columns=["pickup_datetime"])
corr = num_df.corr()

plt.figure(figsize=(11, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Correlation Matrix")
plt.show()

In [ ]:
print("Correlation of each feature with fare_amount:\n")
print(corr["fare_amount"].sort_values(ascending=False).round(3))

In [ ]:
sample = df.sample(min(5000, len(df)), random_state=42)
plt.figure(figsize=(7, 5))
sns.scatterplot(data=sample, x="distance_km", y="fare_amount", alpha=0.4)
plt.title("Distance vs Fare")
plt.show()

# Task 4: Implement Linear, Ridge and Lasso Regression

### 4.1 Select features and split the data

In [ ]:
features = ["pickup_longitude", "pickup_latitude", "dropoff_longitude", "dropoff_latitude",
            "passenger_count", "year", "month", "hour", "day_of_week", "distance_km"]

X = df[features]
y = df["fare_amount"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train:", X_train.shape, " Test:", X_test.shape)

### 4.2 Scale the features
Ridge and Lasso penalize coefficient size, so features must be on the same scale.

In [ ]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

### 4.3 Train the models

In [ ]:
lin = LinearRegression()
ridge = Ridge(alpha=1.0)
lasso = Lasso(alpha=0.01)

lin.fit(X_train_s, y_train)
ridge.fit(X_train_s, y_train)
lasso.fit(X_train_s, y_train)

print("Models trained.")

### 4.4 Compare the learned coefficients

In [ ]:
coef_df = pd.DataFrame({
    "Feature": features,
    "Linear": lin.coef_,
    "Ridge": ridge.coef_,
    "Lasso": lasso.coef_
}).round(4)
coef_df

Lasso can shrink unimportant coefficients to exactly zero (feature selection), while Ridge only shrinks them toward zero.

# Task 5: Evaluate and Compare the Models

In [ ]:
def evaluate(name, model, Xtr, ytr, Xte, yte):
    pred_tr = model.predict(Xtr)
    pred_te = model.predict(Xte)
    return {
        "Model": name,
        "Train R2": r2_score(ytr, pred_tr),
        "Test R2": r2_score(yte, pred_te),
        "MAE": mean_absolute_error(yte, pred_te),
        "MSE": mean_squared_error(yte, pred_te),
        "RMSE": np.sqrt(mean_squared_error(yte, pred_te)),
    }

results = pd.DataFrame([
    evaluate("Linear Regression", lin, X_train_s, y_train, X_test_s, y_test),
    evaluate("Ridge Regression", ridge, X_train_s, y_train, X_test_s, y_test),
    evaluate("Lasso Regression", lasso, X_train_s, y_train, X_test_s, y_test),
]).set_index("Model").round(4)

results

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

results["Test R2"].plot(kind="bar", ax=axes[0], color=["steelblue", "darkorange", "seagreen"])
axes[0].set_title("Test R2 Score (higher is better)")
axes[0].set_ylabel("R2")
axes[0].tick_params(axis="x", rotation=15)

results["RMSE"].plot(kind="bar", ax=axes[1], color=["steelblue", "darkorange", "seagreen"])
axes[1].set_title("RMSE (lower is better)")
axes[1].set_ylabel("RMSE ($)")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.show()

### Actual vs Predicted fares

In [ ]:
models = {"Linear": lin, "Ridge": ridge, "Lasso": lasso}
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for ax, (name, m) in zip(axes, models.items()):
    pred = m.predict(X_test_s)
    ax.scatter(y_test, pred, alpha=0.2, s=8)
    lims = [0, max(y_test.max(), pred.max())]
    ax.plot(lims, lims, "r--")
    ax.set_title(f"{name}: Actual vs Predicted")
    ax.set_xlabel("Actual Fare")
    ax.set_ylabel("Predicted Fare")

plt.tight_layout()
plt.show()

In [ ]:
best = results["Test R2"].idxmax()
print("Best model by Test R2:", best)
print(results.loc[best])

## Conclusion
- The data was cleaned by dropping unused columns, removing missing and duplicate rows, parsing the datetime into year, month, hour and day of week, and computing trip distance with the Haversine formula.
- Outliers (negative or zero fares, invalid coordinates, impossible passenger counts and extreme distances) were identified with boxplots and the IQR rule, then removed.
- The correlation matrix shows `distance_km` has the strongest relationship with `fare_amount`.
- Linear, Ridge and Lasso regression give very similar R2 and RMSE values here, because the features are few and not heavily collinear, so regularization adds little. Lasso may set weak features to zero.
- A linear model cannot fully capture the non-linear parts of fare pricing (traffic, surge, tolls), which limits R2. Tree-based models such as Random Forest could improve on this.